#필수 라이브러리

In [ ]:
%pip install langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 5.0 MB/s eta 0:00:00


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 30.6 MB/s eta 0:00:00


In [ ]:
# from openai import openai
import openai
import os

# API키 불러오기 방법(1) : 코드에 직접 입력하기
# os.environ["OPENAI_API_KEY"] = "여기에 키를 입력한다"

# API키 불러오기 방법(2) : 텍스트 파일에서 읽어오기
# with open('api키를텍스트파일로저장.txt', 'r') as file:
#    os.environ["OPENAI_API_KEY"] = file.read().strip()

# API키 불러오기 방법(3) : 구글 코랩 보안 비밀(Secrets) 기능 이용
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('openai')  # 'openai' 라는 이름으로 저장해둔 API 키를 불러와 환경 변수로 등록


In [ ]:
%%writefile app.py

import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
import base64
import os
from langchain_core.tools import tool
from datetime import datetime
import pytz

llm = ChatOpenAI(model="gpt-4o-mini")

#도구 함수 정의(여기 tool만 쓰면 될 듯!)
@tool


tools = []
tool_dict = {}
llm_with_tools = llm.bind_tools(tools)

# 사용자의 메시지 처리하기 위한 함수
def get_ai_response(messages):
    response = llm_with_tools.stream(messages)

    # 스트리밍된 AIMessageChunk들을 하나의 완성된 메시지로 합치기 위한 변수
    gathered = None

    for chunk in response:
        yield chunk

        if gathered is None:
            gathered = chunk
        else:
            gathered += chunk

    # 완성된 메시지에 Tool Call이 있다면 실제 Tool을 실행
    if gathered.tool_calls:  # TODO ⑩: 완성된 메시지에 Tool 호출 요청이 있는지 확인하는 속성
        st.session_state.messages.append(gathered)

        for tool_call in gathered.tool_calls:
            selected_tool = tool_dict[tool_call["name"]]
            tool_msg = selected_tool.invoke(tool_call)

            print(tool_msg, type(tool_msg))

            st.session_state.messages.append(tool_msg)

        # ToolMessage까지 추가된 새 대화 기록으로 모델을 다시 호출해 최종 답변 생성
        for chunk in get_ai_response(st.session_state.messages):
            yield chunk

# Streamlit 앱
st.set_page_config(
    page_title = "💬 딸깍콘",
    page_icon = "/content/화면 캡처 2026-10-08 183640.png",
    layout = 'wide'
)

uploaded_file = None

with st.sidebar:
  st.header("🤔 이모티콘으로 만들 이미지")
  st.divider()
  uploaded_file = st.file_uploader("캐릭터 이미지 업로드", type = ["png", "jpg", "jpeg", "webp"])

  if uploaded_file is not None:
        st.image(uploaded_file, caption="업로드된 이미지", use_container_width=True)

  st.divider()
  st.header("🎁 완성된 이모티콘")

  gif_path = "/content/generated_emoticon.gif"

  if os.path.exists(gif_path):
    st.image(gif_path, caption="360x360 무한루프 이모티콘", use_container_width=True)

    with open(gif_path, "rb") as file:
      st.download_button(
          label="📥 이모티콘 다운로드 (.gif)",
          data=file,
          file_name="kakao_emoticon.gif",
          mime="image/gif",
          use_container_width=True
      )
  else:
    st.info("💡 채팅창에서 이모티콘 생성을 요청하면 이곳에 다운로드 버튼이 나타납니다!")

st.markdown("""
<style>
/* ① 전체 앱의 배경색을 살짝 밝은 회색/아이보리 톤으로 변경 */
.stApp {
    background-color: #F8F9FA;
}

    /* ② 버튼(st.button) 모양과 색상 고급화 */
    div.stButton > button {
        background-color: #4F46E5 !important;  /* 버튼 배경색 (트렌디한 인디고 보라색) */
        color: white !important;                /* 글자 색상 */
        border-radius: 12px !important;         /* 모서리를 둥글게 (12px) */
        border: none !important;                /* 테두리선 제거 */
        padding: 8px 16px !important;           /* 버튼 내부 여백 */
        font-weight: bold !important;           /* 글자 두껍게 */
        transition: all 0.3s ease !important;   /* 마우스 올렸을 때 부드러운 애니메이션 효과 */
    }

    /* 마우스를 버튼 위에 올렸을 때(Hover) 변하는 효과 */
    div.stButton > button:hover {
        background-color: #3730A3 !important;  /* 더 짙은 보라색으로 변경 */
        transform: translateY(-2px);           /* 버튼이 살짝 위로 붕 뜨는 효과 */
        box-shadow: 0px 4px 12px rgba(79, 70, 229, 0.3) !important; /* 부드러운 그림자 생성 */
    }

    /* ③ 카드 박스(st.metric) 영역 커스텀 */
    div[data-testid="stMetric"] {
        background-color: white !important;
        padding: 20px !important;
        border-radius: 16px !important;         /* 알약처럼 둥근 모서리 */
        box-shadow: 0px 4px 15px rgba(0, 0, 0, 0.05) !important; /* 입체감 나는 은은한 그림자 */
        border: 1px solid #E5E7EB !important;   /* 은은한 회색 테두리 */
    }

    /* ④ 나만의 커스텀 태그(HTML 클래스) 정의하기 */
    .highlight-title {
        color: #1E293B;
        font-size: 2.2rem;
        font-weight: 800;
        margin-bottom: 0px;
    }
    .custom-badge {
        background-color: #E0E7FF;
        color: #3730A3;
        padding: 4px 12px;
        border-radius: 20px;
        font-size: 0.85rem;
        font-weight: 600;
        display: inline-block;
        margin-bottom: 15px;
    }
    </style>
""", unsafe_allow_html = True)
st.markdown('<div class="custom-badge">🤣 딸깍!으로 만드는 이모티콘</div>', unsafe_allow_html = True)
st.markdown('<h1 class="highlight-title">💬딸깍콘', unsafe_allow_html = True)

# 스트림릿 session_state에 메시지 저장
if "messages" not in st.session_state:
    st.session_state["messages"] = [
        SystemMessage(content = """[Role & Goal]
당신은 사용자의 요청을 바탕으로 세심하고 매력적인 '움직이는 이모티콘(Animated Emoticon)'을 기획하고 생성해 주는 친절한 AI 전문 디자이너입니다. 사용자의 아이디어를 100% 존중하며, 구체적이고 완성도 높은 애니메이션 이모티콘 결과를 도출하는 것이 당신의 목적입니다.

[Persona & Tone]
- 항상 밝고 다정하며 친근한 어조로 대화합니다. (예: "~해요!", "~해드릴게요 😊")
- 사용자의 아이디어를 적극적으로 칭찬하고 긍정적인 반응을 보입니다.
- 디자이너의 전문성을 갖추되, 전문 용어보다는 누구나 이해하기 쉽고 따뜻한 표현을 사용합니다.

[Core Instructions]
1. 사용자 프롬프트 최우선 반영:
   - 사용자가 제시한 캐릭터 외형, 동작, 감정, 색상, 분위기, 텍스트 등의 모든 조건(키워드)을 단 하나도 놓치지 않고 반영합니다.
   - 만약 사용자의 요청이 구체적이지 않다면, 의도를 해치지 않는 선에서 자연스럽고 매력적인 애니메이션 연출 아이디어를 친절하게 더해줍니다.

2. 움직이는 이모티콘 연출 상세화:
   - 캐릭터 디자인: 외형, 색상, 분위기
   - 주요 감정 및 표정: 눈빛, 입 모양, 얼굴 각도 변화
   - 프레임별 움직임(Loop): 시작 동작 -> 핵심 액션 -> 반복 연결 동작으로 구분하여 3~4단계로 생생하게 묘사
   - 시각 효과: 하트, 반짝이, 땀방울, 텍스트 연출 등 부가적 모션 효과 추가

3. 이미지 및 애니메이션 생성용 프롬프트 작성:
   - 사용자 기획안을 바탕으로 DALL-E, Midjourney, Stable Diffusion 등 이미지 생성 AI에 입력할 최적의 영문 프롬프트(Prompt)를 함께 제공합니다.
   - 키워드 구성: [Character Details], [Action/Motion sequence], [Expression], [Style: 2D cute emoticon style, transparent background, sticker, smooth animation frames], [Color palette].

[Output Format]
1. 친절한 인사 및 사용자의 아이디어에 대한 긍정적 공감
2. 이모티콘 상세 기획 (캐릭터 특징, 표정, 프레임별 움직임, 부가 효과)
3. AI 이미지/애니메이션 생성용 영문 프롬프트 (코드 블록 제공)""")]

def display_message_content(content):
    if isinstance(content, list):
        for item in content:
            if isinstance(item, dict):
                if item.get("type") == "text":
                    st.write(item.get("text", ""))
                elif item.get("type") == "image_url":
                    img_url = item.get("image_url", {}).get("url", "")
                    if img_url:
                        st.image(img_url, caption="첨부한 이미지", width=200)

# 스트림릿 화면에 메시지 출력
for msg in st.session_state.messages:
    if msg.content:
        if isinstance(msg, SystemMessage):

            continue

        elif isinstance(msg, AIMessage):
            st.chat_message("assistant").write(msg.content)

        elif isinstance(msg, HumanMessage):
            st.chat_message("user").write(msg.content)

        elif isinstance(msg, ToolMessage):  # TODO ⑪: Tool 실행 결과를 나타내는 LangChain 메시지 클래스
            st.chat_message("tool").write(msg.content)

# 사용자 입력 처리
if prompt := st.chat_input():
    if uploaded_file is not None:
        bytes_data = uploaded_file.getvalue()
        base64_image = base64.b64encode(bytes_data).decode('utf-8')
        mime_type = uploaded_file.type

        user_content = [
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{mime_type};base64,{base64_image}"
                }
            }
        ]
        user_message = HumanMessage(content=user_content)
    else:
        user_message = HumanMessage(content=prompt)

    # 화면에 사용자 질문 출력
    with st.chat_message("user"):
        st.write(prompt)

    st.session_state.messages.append(user_message)

    # AI 응답 받아오기
    with st.chat_message("assistant"):
        response = get_ai_response(st.session_state["messages"])
        result = st.write_stream(response)

    st.session_state["messages"].append(
        AIMessage(content=result)
    )


Overwriting app.py


In [ ]:
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!./cloudflared tunnel --url http://localhost:8501

2026-10-08T12:55:35Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-08T12:55:35Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-08T12:55:38Z INF +--------------------------------------------------------------------------------------------+
2026-10-08T12:55:38Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-08T12:55:38Z INF |  https://lap-selecting-lexmark-sum.trycloudflare.com  

추가로 불러오면 좋은것

* 배경제거:rembg

* GIF반환: imageio

* 이미지 편집:pillow

* 영상프레임 처리: imageio-ffmpeg

* 네트워크 통신?: request

# 함수 코드

#